### Импорты

In [6]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [7]:
import sys
from pathlib import Path

root_path = Path.cwd().parent
if str(root_path) not in sys.path:
    sys.path.append(str(root_path))

from src.download import download_and_extract_data

In [8]:
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
from catboost import CatBoostRegressor
from pathlib import Path
from models.classes.baselines import QuantityBaselineModel, MeanIntervalBaselineModel, EmaIntervalBaselineModel
from src.metrics_calculation import calculate_metrics
from src.models_comparing import load_and_evaluate_all_models, display_metrics_by_group
from models.classes.catboost_rourting import CatBoostRegressorRouting

### Глобальные константы

In [9]:
dataset_folder_path = Path("./data/Dunnhumby")
files_folder_path = Path("./data/files")
models_folder_path = Path("./models/fitted")
results_folder_path = Path("./models/results")

quantity_baseline = QuantityBaselineModel()
meanInterval_baseline = MeanIntervalBaselineModel()
emaInterval_baseline = EmaIntervalBaselineModel()

baselines = {
    "Quantity Baseline": quantity_baseline,
    "Mean Baseline": meanInterval_baseline,
    "EMA Baseline": emaInterval_baseline
}

cat_features_list = ['day_of_week']

model_name = "catboost_mape_depth10"

### Загрузка данных

In [26]:
dataset_train = pd.read_csv(dataset_folder_path / "dataset_train.csv")
dataset_val = pd.read_csv(dataset_folder_path / "dataset_val.csv")
dataset_test = pd.read_csv(dataset_folder_path / "dataset_test.csv")

In [27]:
X_train, y_train = dataset_train.drop("target", axis=1), dataset_train["target"]
X_val, y_val = dataset_val.drop("target", axis=1), dataset_val["target"]
X_test, y_test = dataset_test.drop("target", axis=1), dataset_test["target"]

In [28]:
WINDOW_SIZE = 5

In [29]:
# Задаем жесткий лимит: таргет не должен превышать средний интервал более чем в 3.5 раза
# (Добавляем +2 дня для защиты от нулей и очень коротких интервалов)
def clean_targets(df):
    # Оставляем только те строки, где таргет адекватен истории
    mask = df['target'] <= (df['mean_interval'] * 2 + 1)
    
    # Также отрежем абсолютных "черных лебедей" (таргеты > 60 дней), 
    # так как мы договорились, что расходники столько не живут
    mask = mask & (df['target'] <= 60)
    
    print(f"Удалено аномальных таргетов: {len(df) - mask.sum()} из {len(df)}")
    return df[mask]

print("Очистка выбросов в таргетах...")
dataset_train = clean_targets(dataset_train)
dataset_val = clean_targets(dataset_val)
dataset_test = clean_targets(dataset_test)

# И только теперь делим на X и y
X_train, y_train = dataset_train.drop("target", axis=1), dataset_train["target"]
X_val, y_val = dataset_val.drop("target", axis=1), dataset_val["target"]
X_test, y_test = dataset_test.drop("target", axis=1), dataset_test["target"]

Очистка выбросов в таргетах...
Удалено аномальных таргетов: 30466 из 258135
Удалено аномальных таргетов: 6410 из 55807
Удалено аномальных таргетов: 6323 из 54478


### Обучение моделей

In [30]:
# Задаем веса: чем меньше средний интервал, тем важнее пример
# + 1e-5 для защиты от нулей
train_weights = 1.0 / (X_train['mean_interval'] + 1e-5)
val_weights = 1.0 / (X_val['mean_interval'] + 1e-5)

cb_weighted = CatBoostRegressor(
    iterations=2000,
    learning_rate=0.05,
    depth=7,
    loss_function='MAE',
    eval_metric='MAE',
    cat_features=cat_features_list,
    random_seed=42,
    verbose=500
)
print("Обучение CatBoost_Weighted...")
cb_weighted.fit(
    X_train, y_train, 
    sample_weight=train_weights, # <-- Передаем веса
    eval_set=(X_val, y_val), 
    early_stopping_rounds=50, 
    use_best_model=True
)
joblib.dump(cb_weighted, models_folder_path / "CatBoost_Weighted.joblib")

Обучение CatBoost_Weighted...
0:	learn: 6.0296782	test: 9.2927523	best: 9.2927523 (0)	total: 91.1ms	remaining: 3m 2s
500:	learn: 4.3928081	test: 6.7391103	best: 6.7391028 (499)	total: 44.5s	remaining: 2m 13s
1000:	learn: 4.3543278	test: 6.7328509	best: 6.7328509 (1000)	total: 1m 28s	remaining: 1m 28s
1500:	learn: 4.3268633	test: 6.7287394	best: 6.7287121 (1496)	total: 2m 14s	remaining: 44.6s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 6.728037945
bestIteration = 1649

Shrink model to first 1650 iterations.


['models\\fitted\\CatBoost_Weighted.joblib']

### Анализ метрик

In [31]:
all_results_df = load_and_evaluate_all_models(models_folder_path, X_test, y_test, WINDOW_SIZE, baselines)

# Смотрим результаты в разрезе MAPE
display_metrics_by_group(all_results_df, metric_type='mape')

# Смотрим результаты в разрезе MAE
display_metrics_by_group(all_results_df, metric_type='mae')

# Смотрим результаты в разрезе MASE
display_metrics_by_group(all_results_df, metric_type='mase')

Загрузка модели: catboost_mae...
Загрузка модели: catboost_mape...
Загрузка модели: catboost_mape_depth10...
Загрузка модели: CatBoost_Routing_MAE...
Загрузка модели: CatBoost_Routing_MAPE...
Загрузка модели: CatBoost_Routing_Weighted_MAPE...
Загрузка модели: CatBoost_Weighted...

Расчет метрик...

==================== СРАВНЕНИЕ ПО MAPE ====================


,overall,overall_fast,overall_medium,overall_long,cold_start,cold_start_fast,cold_start_medium,cold_start_long,warm_state,warm_state_fast,warm_state_medium,warm_state_long
model_name,,,,,,,,,,,,
Quantity Baseline,1.1685,1.6708,0.9083,1.6708,1.3984,1.8996,0.9783,1.8996,1.1016,1.5723,0.8922,1.5723
Mean Baseline,1.1251,1.6356,0.8607,1.6356,1.3826,1.9100,0.9406,1.9100,1.0502,1.5173,0.8424,1.5173
EMA Baseline,1.1356,1.6269,0.8811,1.6269,1.3766,1.9020,0.9364,1.9020,1.0655,1.5084,0.8684,1.5084
catboost_mae,0.9323,1.1899,0.7989,1.1899,1.0876,1.2014,0.9922,1.2014,0.8872,1.1849,0.7547,1.1849
catboost_mape,0.6331,0.6884,0.6044,0.6884,0.6681,0.6901,0.6497,0.6901,0.6229,0.6876,0.5941,0.6876
catboost_mape_depth10,0.6320,0.6880,0.6030,0.6880,0.6696,0.6904,0.6523,0.6904,0.6211,0.6870,0.5917,0.6870
CatBoost_Routing_MAE,0.9344,1.1903,0.8019,1.1903,1.0885,1.1990,0.9959,1.1990,0.8896,1.1865,0.7575,1.1865
CatBoost_Routing_MAPE,0.6161,0.6866,0.5795,0.6866,0.6488,0.6881,0.6157,0.6881,0.6066,0.6860,0.5712,0.6860
CatBoost_Routing_Weighted_MAPE,0.6191,0.6834,0.5858,0.6834,0.6514,0.6872,0.6214,0.6872,0.6097,0.6818,0.5776,0.6818



==================== СРАВНЕНИЕ ПО MAE ====================


,overall,overall_fast,overall_medium,overall_long,cold_start,cold_start_fast,cold_start_medium,cold_start_long,warm_state,warm_state_fast,warm_state_medium,warm_state_long
model_name,,,,,,,,,,,,
Quantity Baseline,9.5289,17.3037,5.5010,17.3037,12.0584,19.6021,5.7372,19.6021,8.7928,16.3134,5.4469,16.3134
Mean Baseline,8.3887,15.4472,4.7318,15.4472,11.2071,18.3804,5.1962,18.3804,7.5685,14.1833,4.6256,14.1833
EMA Baseline,8.8913,16.1717,5.1194,16.1717,11.2647,18.4514,5.2425,18.4514,8.2005,15.1894,5.0912,15.1894
catboost_mae,7.0742,11.9197,4.5638,11.9197,8.4172,12.2599,5.1972,12.2599,6.6833,11.7731,4.4190,11.7731
catboost_mape,9.3359,17.3042,5.2077,17.3042,11.0724,17.6433,5.5663,17.6433,8.8305,17.1580,5.1256,17.1580
catboost_mape_depth10,9.3322,17.2987,5.2049,17.2987,11.0795,17.6517,5.5723,17.6517,8.8237,17.1466,5.1208,17.1466
CatBoost_Routing_MAE,7.0839,11.9249,4.5759,11.9249,8.4207,12.2489,5.2128,12.2489,6.6949,11.7853,4.4302,11.7853
CatBoost_Routing_MAPE,9.3402,16.6865,5.5343,16.6865,10.9756,16.9832,5.9415,16.9832,8.8643,16.5586,5.4411,16.5586
CatBoost_Routing_Weighted_MAPE,9.1316,16.1090,5.5167,16.1090,10.7359,16.5027,5.9036,16.5027,8.6647,15.9394,5.4282,15.9394



==================== СРАВНЕНИЕ ПО MASE ====================


,overall,overall_fast,overall_medium,overall_long,cold_start,cold_start_fast,cold_start_medium,cold_start_long,warm_state,warm_state_fast,warm_state_medium,warm_state_long
model_name,,,,,,,,,,,,
Quantity Baseline,0.8729,0.8787,0.8637,0.8787,0.9292,0.9300,0.9269,0.9300,0.8523,0.8543,0.8497,0.8543
Mean Baseline,0.7685,0.7844,0.7429,0.7844,0.8636,0.8720,0.8395,0.8720,0.7336,0.7427,0.7216,0.7427
EMA Baseline,0.8145,0.8212,0.8038,0.8212,0.8680,0.8754,0.8470,0.8754,0.7949,0.7954,0.7942,0.7954
catboost_mae,0.6480,0.6053,0.7165,0.6053,0.6486,0.5816,0.8397,0.5816,0.6478,0.6165,0.6893,0.6165
catboost_mape,0.8552,0.8787,0.8176,0.8787,0.8532,0.8371,0.8993,0.8371,0.8560,0.8985,0.7996,0.8985
catboost_mape_depth10,0.8549,0.8784,0.8172,0.8784,0.8538,0.8375,0.9003,0.8375,0.8553,0.8979,0.7988,0.8979
CatBoost_Routing_MAE,0.6489,0.6055,0.7184,0.6055,0.6489,0.5811,0.8422,0.5811,0.6489,0.6171,0.6911,0.6171
CatBoost_Routing_MAPE,0.8556,0.8473,0.8689,0.8473,0.8458,0.8057,0.9599,0.8057,0.8592,0.8671,0.8488,0.8671
CatBoost_Routing_Weighted_MAPE,0.8365,0.8180,0.8662,0.8180,0.8273,0.7829,0.9538,0.7829,0.8399,0.8347,0.8468,0.8347


In [32]:
# Ищем строки, где исторически покупали часто (<=7), но таргет оказался огромным (>21)
anomalies = dataset_test[(dataset_test['mean_interval'] <= 7) & (dataset_test['target'] > 21)]

print(f"Количество аномалий в тесте: {len(anomalies)}")
display(anomalies[['session_step', 'mean_interval', 'target']].head(10))

Количество аномалий в тесте: 0


,session_step,mean_interval,target
